# ⚡ Gemini-Optimized Azure ML Forecasting Pipeline
**DP-100 Exam Prep | Cost-Efficient AutoML with Gemini Pro**

---
| Setting | Value |
|---|---|
| Subscription | `` |
| Resource Group | `` |
| Workspace | `` |
| Region | `` |

---
## Pipeline Flow
```
[Open Source Data] → [Prep] → [🤖 Gemini Optimizer] → [Train] → [Model Output]
```

## 📦 Step 1: Install Dependencies

In [ ]:
%pip install azure-ai-ml azure-identity azure-mgmt-resource requests --quiet
print('✅ Dependencies installed')

## 🔗 Step 2: Connect to Azure ML Workspace
> **Always run this cell first after any kernel restart**

In [ ]:
from azure.ai.ml import MLClient, Input, Output, command
from azure.ai.ml.dsl import pipeline
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import Data
from azure.identity import DefaultAzureCredential
import os, json, pandas as pd, numpy as np
import matplotlib.pyplot as plt

ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id='',
    resource_group_name='',
    workspace_name=''
)

ws = ml_client.workspaces.get('')
print(f'✅ Connected: {ws.name}')
print(f'📍 Region:    {ws.location}')
print(f'📁 RG:        {ws.resource_group}')

## 🔑 Step 3: Set Gemini API Key
> Get your key from https://aistudio.google.com/app/apikey

In [ ]:
import os

os.environ['GEMINI_API_KEY'] = 'paste-your-gemini-api-key-here'

api_key = os.environ.get('GEMINI_API_KEY')
if api_key and api_key != 'paste-your-gemini-api-key-here':
    print(f'✅ API key set: {api_key[:8]}...')
else:
    print('❌ Please paste your actual Gemini API key above!')

## 📊 Step 4: Generate & Register Dataset
> Simulates realistic daily energy consumption with yearly/weekly seasonality

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

# ── Generate realistic energy dataset ───────────────────
np.random.seed(42)
dates  = pd.date_range(start='2018-01-01', end='2023-12-31', freq='D')
t      = np.arange(len(dates))
energy = (12000
          + 2000 * np.sin(2 * np.pi * t / 365.25)   # yearly seasonality
          + 500  * np.sin(2 * np.pi * t / 7)         # weekly pattern
          + 0.5  * t                                  # upward trend
          + np.random.normal(0, 300, len(dates)))     # noise

df = pd.DataFrame({
    'date':               dates,
    'energy_consumption': np.round(energy, 2)
})

print(f'✅ Dataset: {df.shape[0]} rows')
print(f'📅 Range:   {df["date"].min().date()} → {df["date"].max().date()}')
print(df.describe())

# ── Plot ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 7))
axes[0].plot(df['date'], df['energy_consumption'], color='steelblue', linewidth=0.8)
axes[0].set_title('Full Time Series — Daily Energy Consumption')
axes[0].set_ylabel('Energy (MW)')

monthly = df.groupby(df['date'].dt.to_period('M'))['energy_consumption'].mean()
monthly.plot(ax=axes[1], color='orange', linewidth=1.5)
axes[1].set_title('Monthly Average')
axes[1].set_ylabel('Energy (MW)')
plt.tight_layout()
plt.show()

# ── Save & register ──────────────────────────────────────
os.makedirs('./data', exist_ok=True)
df.to_csv('./data/energy_timeseries.csv', index=False)

data_asset = ml_client.data.create_or_update(
    Data(
        path='./data/energy_timeseries.csv',
        type=AssetTypes.URI_FILE,
        name='energy-timeseries',
        description='Simulated daily energy consumption - DP100 demo'
    )
)
print(f'✅ Registered: {data_asset.name} v{data_asset.version}')

## 🛠️ Step 5: Create Pipeline Component Scripts
> Writes 3 Python scripts to disk: `prep.py`, `optimize.py`, `train.py`

In [ ]:
import os

os.makedirs('./components/prep',             exist_ok=True)
os.makedirs('./components/train',            exist_ok=True)
os.makedirs('./components/gemini_optimizer', exist_ok=True)

# ── prep.py ──────────────────────────────────────────────
prep_script = '''
import argparse, pandas as pd, os

parser = argparse.ArgumentParser()
parser.add_argument('--raw_data',   type=str)
parser.add_argument('--train_data', type=str)
parser.add_argument('--test_data',  type=str)
args, _ = parser.parse_known_args()

df = pd.read_csv(args.raw_data)
df['date'] = pd.to_datetime(df['date'])
df = df.dropna().sort_values('date')

split = int(len(df) * 0.8)
train, test = df[:split], df[split:]

os.makedirs(args.train_data, exist_ok=True)
os.makedirs(args.test_data,  exist_ok=True)
train.to_csv(os.path.join(args.train_data, 'train.csv'), index=False)
test.to_csv(os.path.join(args.test_data,   'test.csv'),  index=False)

print(f'✅ Train: {len(train)} rows | Test: {len(test)} rows')
'''

# ── optimize.py ──────────────────────────────────────────
optimize_script = '''
import argparse, pandas as pd, json, os, requests

parser = argparse.ArgumentParser()
parser.add_argument('--train_data',       type=str)
parser.add_argument('--optimized_params', type=str)
args, _ = parser.parse_known_args()

api_key = os.environ.get('GEMINI_API_KEY')
if not api_key:
    raise ValueError('GEMINI_API_KEY not set!')
print('✅ Gemini API key loaded')

df = pd.read_csv(os.path.join(args.train_data, 'train.csv'))
df['date'] = pd.to_datetime(df['date'])

profile = {
    'rows':            len(df),
    'date_range_days': (df['date'].max() - df['date'].min()).days,
    'mean':            round(df['energy_consumption'].mean(), 2),
    'std':             round(df['energy_consumption'].std(), 2),
    'min':             round(df['energy_consumption'].min(), 2),
    'max':             round(df['energy_consumption'].max(), 2),
    'trend':           'increasing' if df['energy_consumption'].iloc[-30:].mean() >
                                       df['energy_consumption'].iloc[:30].mean()
                                    else 'decreasing'
}
print(f'📊 Profile: {json.dumps(profile, indent=2)}')

prompt = f\'\'\'You are an expert ML engineer optimizing a time series pipeline.
Dataset: {json.dumps(profile)}
Respond ONLY with valid JSON with keys:
n_estimators (50-500), max_depth (2-8), learning_rate (0.01-0.3),
forecast_horizon (int), lag_features (list), rolling_windows (list), reasoning (str)\'\'\'

response = requests.post(
    'https://generativelanguage.googleapis.com/v1beta/models/gemini-pro:generateContent',
    headers={'Content-Type': 'application/json'},
    params={'key': api_key},
    json={'contents': [{'parts': [{'text': prompt}]}]}
)

raw    = response.json()
text   = raw['candidates'][0]['content']['parts'][0]['text']
text   = text.strip().replace('```json','').replace('```','').strip()
params = json.loads(text)

print(f'🤖 Gemini params: {json.dumps(params, indent=2)}')

os.makedirs(args.optimized_params, exist_ok=True)
with open(os.path.join(args.optimized_params, 'params.json'), 'w') as f:
    json.dump(params, f, indent=2)
print('✅ Optimized params saved')
'''

# ── train.py ─────────────────────────────────────────────
train_script = '''
import argparse, pandas as pd, pickle, os, json
from sklearn.ensemble import GradientBoostingRegressor

parser = argparse.ArgumentParser()
parser.add_argument('--train_data',       type=str)
parser.add_argument('--optimized_params', type=str)
parser.add_argument('--model_output',     type=str)
args, _ = parser.parse_known_args()

with open(os.path.join(args.optimized_params, 'params.json')) as f:
    params = json.load(f)
print(f'🤖 Using Gemini params: {params}')

df = pd.read_csv(os.path.join(args.train_data, 'train.csv'))
df['date']      = pd.to_datetime(df['date'])
df['dayofweek'] = df['date'].dt.dayofweek
df['month']     = df['date'].dt.month
df['dayofyear'] = df['date'].dt.dayofyear

for lag in params['lag_features']:
    df[f'lag_{lag}'] = df['energy_consumption'].shift(lag)

for window in params['rolling_windows']:
    df[f'rolling_{window}'] = df['energy_consumption'].rolling(window).mean()

df = df.dropna()
feature_cols = (
    ['dayofweek','month','dayofyear'] +
    [f'lag_{l}'     for l in params['lag_features']] +
    [f'rolling_{w}' for w in params['rolling_windows']]
)

model = GradientBoostingRegressor(
    n_estimators  = params['n_estimators'],
    max_depth     = params['max_depth'],
    learning_rate = params['learning_rate'],
    random_state  = 42
)
model.fit(df[feature_cols], df['energy_consumption'])

os.makedirs(args.model_output, exist_ok=True)
with open(os.path.join(args.model_output, 'model.pkl'), 'wb') as f:
    pickle.dump(model, f)
with open(os.path.join(args.model_output, 'features.json'), 'w') as f:
    json.dump(feature_cols, f)

print('✅ Model trained & saved')
'''

with open('./components/prep/prep.py',                   'w') as f: f.write(prep_script)
with open('./components/gemini_optimizer/optimize.py',   'w') as f: f.write(optimize_script)
with open('./components/train/train.py',                 'w') as f: f.write(train_script)

print('✅ Component scripts created:')
print(f'   prep.py exists:     {os.path.exists("./components/prep/prep.py")}')
print(f'   optimize.py exists: {os.path.exists("./components/gemini_optimizer/optimize.py")}')
print(f'   train.py exists:    {os.path.exists("./components/train/train.py")}')

## 🚀 Step 6: Build & Submit the Pipeline
> Defines all 3 components and submits the full pipeline to Azure ML

In [ ]:
from azure.ai.ml.dsl import pipeline
from azure.ai.ml import Input, Output, command
from azure.ai.ml.constants import AssetTypes
import os

ENV = 'azureml:AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest'

# ── Component 1: Prep ────────────────────────────────────
prep_component = command(
    name='prep_timeseries_data',
    display_name='📦 Prepare Time Series Data',
    inputs={'raw_data': Input(type=AssetTypes.URI_FILE)},
    outputs={
        'train_data': Output(type=AssetTypes.URI_FOLDER),
        'test_data':  Output(type=AssetTypes.URI_FOLDER)
    },
    code='./components/prep',
    command='python prep.py \
             --raw_data ${{inputs.raw_data}} \
             --train_data ${{outputs.train_data}} \
             --test_data ${{outputs.test_data}}',
    environment=ENV
)

# ── Component 2: Gemini Optimizer ────────────────────────
gemini_component = command(
    name='gemini_param_optimizer',
    display_name='🤖 Gemini Pro Parameter Optimizer',
    inputs={'train_data': Input(type=AssetTypes.URI_FOLDER)},
    outputs={'optimized_params': Output(type=AssetTypes.URI_FOLDER)},
    code='./components/gemini_optimizer',
    command='pip install requests && python optimize.py \
             --train_data ${{inputs.train_data}} \
             --optimized_params ${{outputs.optimized_params}}',
    environment=ENV,
    environment_variables={
        'GEMINI_API_KEY': os.environ.get('GEMINI_API_KEY')
    }
)

# ── Component 3: Train ───────────────────────────────────
train_component = command(
    name='train_forecast_model',
    display_name='🏋️ Train Forecasting Model',
    inputs={
        'train_data':       Input(type=AssetTypes.URI_FOLDER),
        'optimized_params': Input(type=AssetTypes.URI_FOLDER)
    },
    outputs={'model_output': Output(type=AssetTypes.URI_FOLDER)},
    code='./components/train',
    command='python train.py \
             --train_data ${{inputs.train_data}} \
             --optimized_params ${{inputs.optimized_params}} \
             --model_output ${{outputs.model_output}}',
    environment=ENV
)

# ── Pipeline definition ──────────────────────────────────
@pipeline(
    display_name='⚡ Gemini-Optimized Energy Forecasting Pipeline',
    description='Cost-efficient pipeline using Gemini Pro for param optimization'
)
def gemini_optimized_pipeline(raw_data: Input):
    prep     = prep_component(raw_data=raw_data)
    optimize = gemini_component(train_data=prep.outputs.train_data)
    train    = train_component(
        train_data=prep.outputs.train_data,
        optimized_params=optimize.outputs.optimized_params
    )
    return {'model_output': train.outputs.model_output}

# ── Fetch data asset ─────────────────────────────────────
data_asset = ml_client.data.get(name='energy-timeseries', label='latest')
print(f'✅ Data asset: {data_asset.name} v{data_asset.version}')

# ── Build & submit ───────────────────────────────────────
pipeline_job = gemini_optimized_pipeline(
    raw_data=Input(
        path=f'azureml:{data_asset.name}:{data_asset.version}',
        type=AssetTypes.URI_FILE
    )
)
pipeline_job.settings.default_compute = 'serverless'

submitted = ml_client.jobs.create_or_update(pipeline_job)
print(f'✅ Pipeline submitted: {submitted.name}')
print(f'📊 Designer URL: {submitted.studio_url}')

## 📈 Step 7: Monitor Pipeline Job

In [ ]:
# Stream live logs
print('📡 Streaming pipeline logs...')
ml_client.jobs.stream(submitted.name)

# Check final status
job = ml_client.jobs.get(submitted.name)
print(f'\n🏁 Final status: {job.status}')

## 🏆 Step 8: Register the Trained Model

In [ ]:
from azure.ai.ml.entities import Model

model = ml_client.models.create_or_update(
    Model(
        path=f'azureml://jobs/{submitted.name}/outputs/model_output',
        name='gemini-energy-forecast-model',
        description='Gemini-optimized GBR forecasting model',
        type='custom_model'
    )
)
print(f'✅ Model registered: {model.name} v{model.version}')

## 📋 Step 9: List All Assets in Workspace

In [ ]:
print('📋 Data Assets:')
for d in ml_client.data.list():
    print(f'   → {d.name} | v{d.version} | {d.type}')

print('\n🤖 Registered Models:')
for m in ml_client.models.list():
    print(f'   → {m.name} | v{m.version}')

print('\n📦 Datastores:')
for ds in ml_client.datastores.list():
    print(f'   → {ds.name} | {ds.type}')

print('\n⚙️  Jobs:')
for job in list(ml_client.jobs.list())[:5]:
    print(f'   → {job.name} | {job.status}')